In [ ]:
import os
from dotenv import load_dotenv
from model.spark import SparkFactory
from model.minio_store import MinioStore

load_dotenv()

In [ ]:
factory = SparkFactory('ohlcv_analysis')
spark = factory.session

In [ ]:
store = MinioStore(os.getenv('MINIO_ANALYSIS_BUCKET', 'market-analysis'))

parquet_files = [
    f's3a://{store.bucket}/{obj.object_name}'
    for obj in store.list_objects(prefix='ohlcv.bar/')
    if obj.object_name.endswith('.parquet')
]

print(f'Found {len(parquet_files)} file(s)')
for f in parquet_files:
    print(' ', f)

In [ ]:
df = spark.read.parquet(*parquet_files)
df.printSchema()
df.show(truncate=False)

In [ ]:
factory.stop()